In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Table loading
application_features  = spark.table("ml_layer.application_features_clean")

# Stratification split
fraud = application_features.filter("is_fraud = 1")
legit = application_features.filter("is_fraud = 0")

train_fraud, test_fraud = fraud.randomSplit([0.8, 0.2], seed=42)
train_legit, test_legit = legit.randomSplit([0.8, 0.2], seed=42)

train_df = train_fraud.union(train_legit)
test_df = test_fraud.union(test_legit)

# Then I apply StringIndexing for payment_type which is the only categorical variable in the dataset

indexer = StringIndexer(
    inputCol="payment_type", 
    outputCol="payment_type_indexed", 
    handleInvalid="keep")

# Vector Assembling

feature_cols = [
    'log_income',
    'address_stability',
    'under_25',
    'name_email_similarity',
    'days_since_request',
    'payment_type_indexed',
]

assembler = VectorAssembler(
    inputCols = feature_cols,
    outputCol = "features"
)

# And now the pipeline is build up

pipeline = Pipeline(stages=[indexer, assembler])

pipeline_model = pipeline.fit(train_df)

train_final = pipeline_model.transform(train_df).select(
    "account_id",
    "features",
    "is_fraud"
)

test_final = pipeline_model.transform(test_df).select(
    "account_id",
    "features",
    "is_fraud"
)


# Finaly the table is saved

train_final.write.format("delta"). \
    mode("overwrite"). \
    saveAsTable("ml_layer.application_train_features")
    
test_final.write.format("delta"). \
    mode("overwrite"). \
    saveAsTable("ml_layer.application_test_features")